In [1]:
import numpy
import collections 
import json

In [2]:
import dgl
import dgl.nn
import torch as tr
import torch.nn as nn
import torch.nn.functional as F
import itertools

Using backend: pytorch


In [3]:
# Opening JSON file
def load_json(filepath):
    with open(str(filepath)) as json_file:
        json_load = json.load(json_file)
    return json_load



def str_to_func(pkg ,func_name):
    return getattr(pkg, func_name)



#Return layer_list: all layers in sequence, param: a nested dict that {layer:{para: value}}
#TODO: Previous item overwrite previous one!
def parser(json_load):
    layer_list = []
    detail = {}
    param_dict = {}
    
    #Loop for layers
    for l in json_load['layers']:
        layer_list.append(l['type'])
        
        if l['parameters']:
            for p in l['parameters']:
                
                if 'activation' in p.keys() and p['activation']:
                    p['activation'] = str_to_func(F, p['activation'].lower())
                    detail.update(p)
                else:
                    detail.update(p)

                param_dict[l['type']] = detail
        else: 
            param_dict[l['type']] = None

        detail={}
        
    #Loop for layers ends
    
    hyperparam_dict = layers['hyperparameters']

    return layer_list, param_dict, hyperparam_dict


#To construct a dictionary for all dgl.nn obejects; Sample: {string_pattern : certain layer}
def get_layer_dict(layer_list):
    
    layer_dict = collections.OrderedDict()
    lib = dir(dgl.nn)
    
    for l in layer_list:
        for f in lib:
            if str(f) in l: 
                layer_dict[l] = str(f)
    
    return layer_dict


#Input：(layer_dict, param_dict) result of parser
#Return: e.g [nn.Conv2d(5,5,3), nn.BatchNorm2d(5), nn.Linear(5,2)]
def get_layer_list(layer_dict, param_dict):
    layer_list = []
    for x,y in layer_dict.items():
        layer = str_to_func(dgl.nn, y)
        
        if param_dict[x]:
            layer_list.append(layer(**param_dict[x]))
        else:
            layer_list.append(layer())
    return layer_list

In [4]:
#Test
layers = load_json('layers_example.json')
layer_list, param_dict,  hyperparam_dict= parser(layers)
print(layers)
print('\n')
print(layer_list)
print(param_dict)
print('\n')
print(hyperparam_dict)

{'layers': [{'type': 'SGConv01', 'parameters': [{'in_feats': 5}, {'out_feats': 16}, {'bias': True}]}, {'type': 'SAGEConv02', 'parameters': [{'in_feats': 16}, {'out_feats': 2}, {'aggregator_type': 'mean'}]}], 'hyperparameters': {'optimizer': 'Adam', 'epochs': 100, 'lr': 0.01, 'batchs': 1}}


['SGConv01', 'SAGEConv02']
{'SGConv01': {'in_feats': 5, 'out_feats': 16, 'bias': True}, 'SAGEConv02': {'in_feats': 16, 'out_feats': 2, 'aggregator_type': 'mean'}}


{'optimizer': 'Adam', 'epochs': 100, 'lr': 0.01, 'batchs': 1}


In [5]:
d = get_layer_dict(layer_list)
d

OrderedDict([('SGConv01', 'SGConv'), ('SAGEConv02', 'SAGEConv')])

In [6]:
layer_list = get_layer_list(d, param_dict)

# Test on GraphSAGE

In [7]:
from tutorial_utils import load_zachery

# ----------- 0. load graph -------------- #
g = load_zachery()
print(g)

Graph(num_nodes=34, num_edges=156,
      ndata_schemes={'club': Scheme(shape=(), dtype=torch.int64), 'club_onehot': Scheme(shape=(2,), dtype=torch.int64)}
      edata_schemes={})


In [15]:
# ----------- 1. node features -------------- #
node_embed = nn.Embedding(g.number_of_nodes(), 5)  # Every node has an embedding of size 5.
inputs = node_embed.weight                         # Use the embedding weight as the node features.
nn.init.xavier_uniform_(inputs)
#print(inputs)
labels = g.ndata['club']
labeled_nodes = [0, 33]
print('Labels', labels[labeled_nodes])
print('\n')

# ----------- 2. create model -------------- #
# build a two-layer GraphSAGE model
class GraphSAGE(nn.Module):
    def __init__(self, layer_list):
        super(GraphSAGE, self).__init__()
        self.layers = nn.ModuleList(layer_list)
    
    def forward(self, g, in_feat):
        first_layer = self.layers[0]
        h = first_layer(g, in_feat)
        for layer in self.layers[1:]:
            h = layer(g, h)
        return h
    
# Create the model with given dimensions 
# input layer dimension: 5, node embeddings
# hidden layer dimension: 16
# output layer dimension: 2, the two classes, 0 and 1
net = GraphSAGE(layer_list)


# ----------- 3. set up loss and optimizer -------------- #
#Set hyperparameters
optimizer = str_to_func(tr.optim, hyperparam_dict['optimizer'])
lr = hyperparam_dict['lr']
epoch_range = range(hyperparam_dict['epochs'])
batch_size = hyperparam_dict['batchs']
print('Optimizer: ' + str(hyperparam_dict['optimizer']))
print('Learning rate: ' + str(lr))
print('Epoch range: ' + str(hyperparam_dict['epochs']))
print('Batch size: ' + str(hyperparam_dict['batchs']))
print('\n')

optimizer = optimizer(itertools.chain(net.parameters(), node_embed.parameters()), lr=lr)

# ----------- 4. training -------------------------------- #
all_logits = []
for e in epoch_range:
    # forward
    logits = net(g, inputs)
    
    # compute loss
    logp = F.log_softmax(logits, 1)
    loss = F.nll_loss(logp[labeled_nodes], labels[labeled_nodes])
    
    # backward
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    all_logits.append(logits.detach())
    
    if e % 5 == 0:
        print('In epoch {}, loss: {}'.format(e, loss))

Labels tensor([0, 1])


Optimizer: Adam
Learning rate: 0.01
Epoch range: 100
Batch size: 1


In epoch 0, loss: 1.8086446523666382
In epoch 5, loss: 0.14463777840137482
In epoch 10, loss: 0.014779075980186462
In epoch 15, loss: 0.0022172420285642147
In epoch 20, loss: 0.0005988333141431212
In epoch 25, loss: 0.00025190634187310934
In epoch 30, loss: 0.0001433359575457871
In epoch 35, loss: 9.965275239665061e-05
In epoch 40, loss: 7.885312516009435e-05
In epoch 45, loss: 6.782729178667068e-05
In epoch 50, loss: 6.156931340228766e-05
In epoch 55, loss: 5.775490717496723e-05
In epoch 60, loss: 5.5311291362158954e-05
In epoch 65, loss: 5.376167973736301e-05
In epoch 70, loss: 5.262926424620673e-05
In epoch 75, loss: 5.1794857427012175e-05
In epoch 80, loss: 5.107964534545317e-05
In epoch 85, loss: 5.048363527748734e-05
In epoch 90, loss: 4.99472334922757e-05
In epoch 95, loss: 4.947042907588184e-05


In [9]:
#Original Code From https://github.com/dglai/WWW20-Hands-on-Tutorial/blob/master/basic_tasks/2_gnn.ipynb

from dgl.nn import SAGEConv

# ----------- 2. create model -------------- #
# build a two-layer GraphSAGE model
class GraphSAGE(nn.Module):
    def __init__(self, in_feats, h_feats, num_classes):
        super(GraphSAGE, self).__init__()
        self.conv1 = SAGEConv(in_feats, h_feats, 'mean', activation = F.relu)
        self.conv2 = SAGEConv(h_feats, num_classes, 'mean')
    
    def forward(self, g, in_feat):
        h = self.conv1(g, in_feat)
        h = self.conv2(g, h)
        return h
    
# Create the model with given dimensions 
# input layer dimension: 5, node embeddings
# hidden layer dimension: 16
# output layer dimension: 2, the two classes, 0 and 1
net = GraphSAGE(5, 16, 2)

# ----------- 3. set up loss and optimizer -------------- #
# in this case, loss will in training loop
optimizer = tr.optim.Adam(itertools.chain(net.parameters(), node_embed.parameters()), lr=0.01)

# ----------- 4. training -------------------------------- #
all_logits = []
for e in range(100):
    # forward
    logits = net(g, inputs)
    
    # compute loss
    logp = F.log_softmax(logits, 1)
    loss = F.nll_loss(logp[labeled_nodes], labels[labeled_nodes])
    
    # backward
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    all_logits.append(logits.detach())
    
    if e % 5 == 0:
        print('In epoch {}, loss: {}'.format(e, loss))

In epoch 0, loss: 0.770458459854126
In epoch 5, loss: 0.27334555983543396
In epoch 10, loss: 0.10055465996265411
In epoch 15, loss: 0.032324522733688354
In epoch 20, loss: 0.011167950928211212
In epoch 25, loss: 0.004677165299654007
In epoch 30, loss: 0.0024075317196547985
In epoch 35, loss: 0.0014686952345073223
In epoch 40, loss: 0.0010221092961728573
In epoch 45, loss: 0.0007835508440621197
In epoch 50, loss: 0.0006435803952626884
In epoch 55, loss: 0.0005546439788304269
In epoch 60, loss: 0.0004942960222251713
In epoch 65, loss: 0.000450685853138566
In epoch 70, loss: 0.0004172024200670421
In epoch 75, loss: 0.0003902718599420041
In epoch 80, loss: 0.00036769016878679395
In epoch 85, loss: 0.0003481467720121145
In epoch 90, loss: 0.0003307480947114527
In epoch 95, loss: 0.00031495795701630414


# Test On Piredict_num_deges

In [10]:
import dgl
import torch

def collate(samples):
    # The input `samples` is a list of pairs
    #  (graph, label).
    new_samples = []
    for graph, _ in samples:
        # Rather than the number of nodes, the labels changed to number of edges
        new_samples.append((graph, torch.tensor(float(graph.number_of_edges()))))
    graphs, labels = map(list, zip(*new_samples))
     
    batched_graph = dgl.batch(graphs)
    return batched_graph, torch.tensor(labels)

In [11]:

###############################################################################
# Setup and training
# ------------------
# Create a synthetic dataset of :math:`400` graphs with :math:`10` ~
# :math:`20` nodes. :math:`320` graphs constitute a training set and
# :math:`80` graphs constitute a test set.
from dgl.data import MiniGCDataset
import matplotlib.pyplot as plt
import networkx as nx
import torch.optim as optim
from torch.utils.data import DataLoader
from dgl.nn.pytorch import GraphConv
import numpy as np

# Create training and test sets.
trainset = MiniGCDataset(320, 10, 20)
testset = MiniGCDataset(80, 10, 20)
# Use PyTorch's DataLoader and the collate function
# defined before.

# Tunes specific parameters (currently only learning rate and hidden dimensions)
# Outputs the MSE and Epoch losses array
def tuning_round(curr_lr, curr_epoch_range, curr_batch_size):
    # 4/15: Made batch_size to 1
    data_loader = DataLoader(trainset, batch_size=curr_batch_size, shuffle=True,
                             collate_fn=collate)

    # Create model
    #modified to contain a single output
    #model = Classifier(1, 256, trainset.num_classes)
    #TODO: need to train hyperparameters hidden layers
    # hidden_dim = 256
    model = myNet()
    loss_func = nn.MSELoss()

    #modifications to migrate to regression
    #loss_func = nn.CrossEntropyLoss()
    #optimizer = optim.Adam(model.parameters(), lr=0.001)
    optimizer = torch.optim.SGD(model.parameters(), lr=curr_lr)
    model.train()

    epoch_losses = [] # MSE average list
    epoch_r_losses = [] # RMSE average list
    for epoch in range(curr_epoch_range):
        epoch_loss = 0
        epoch_r_loss = 0
        for iter, (bg, label) in enumerate(data_loader):
            
            print(type(bg)) #########################
            
            prediction = model(bg)
            loss = loss_func(prediction, label)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            MRE_loss = np.average(np.abs(np.array(prediction.detach().numpy()) - np.array(label.detach().numpy()))/np.abs(np.array(label.detach().numpy())))
            # Make sure the labels are normal and not irregularly high
            epoch_loss += loss.detach().item()
            if np.isinf(MRE_loss):
                # Occurs when one of the label values is zero
#                 fig, ax = plt.subplots()
#                 nx.draw(graph.to_networkx(), ax=ax)
#                 print("example graph:")
#                 plt.show()
#                 print(graph)
#                 print(label)
                continue
            print(MRE_loss)
            
            
            epoch_r_loss += MRE_loss
        epoch_loss /= (iter + 1)
        epoch_r_loss /= (iter + 1)

#         print('Epoch {}, loss {:.4f}'.format(epoch, epoch_loss))
#         print(prediction)
#         print(np.array(prediction.detach().numpy()) - np.array(label.detach().numpy()))
#         print(loss)
#         curr_MSE = np.average(np.abs(np.array(prediction.detach().numpy()) - np.array(label.detach().numpy()))/abs(np.array(label.detach().numpy())))
#         MSE.append(curr_MSE)

        epoch_losses.append(epoch_loss)
        epoch_r_losses.append(epoch_r_loss)
    return epoch_losses, epoch_r_losses

In [12]:
mse_lr_arr = []
rmse_lr_arr = []
curr_epoch_size = 80
curr_batch_size = 32
for i in [0.01, 0.001, 0.0005, 0.0001]:
    #print(i)
    epoch_losses_arr, epoch_r_losses_arr = tuning_round(i, curr_epoch_size, curr_batch_size)
    mse_lr = sum(epoch_losses_arr)/len(epoch_losses_arr)
    rmse_lr = sum(epoch_r_losses_arr)/len(epoch_r_losses_arr)
    mse_lr_arr.append(mse_lr)
    rmse_lr_arr.append(rmse_lr)
print(mse_lr_arr)
print(rmse_lr_arr)

NameError: name 'myNet' is not defined